In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.stats import entropy
import os
import zipfile
from joblib import Parallel, delayed
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. 閾値特定シミュレーションの設定
# ==========================================
N_TRIALS = 1000
DIMENSIONS = np.arange(10, 160, 20) # Dを細かく設定
SAMPLE_SIZES = np.logspace(3, 6.5, num=30, dtype=int)
EPSILON_THRESHOLD = 0.001
output_dir = "hypothesis3_threshold_analysis"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. 単一試行関数 (H3統合DGP)
# ==========================================
def simulate_fast(D, N):
    K = 5
    cell_raw = np.random.uniform(0, 0.2, D)
    P_cell = cell_raw / np.sum(cell_raw)
    betas = 10 ** np.random.uniform(-1, 1.5, D)
    X_choice = np.random.uniform(0, 0.2, (D, K))

    P_joint = np.zeros((D, K))
    for d in range(D):
        exp_val = np.exp(betas[d] * X_choice[d])
        P_choice_given_d = exp_val / np.sum(exp_val)
        P_joint[d, :] = P_cell[d] * P_choice_given_d

    P_flat = P_joint.flatten()
    C_flat = np.random.multinomial(N, P_flat)
    Q_flat = C_flat / N
    Q_flat = np.where(Q_flat == 0, 1e-10, Q_flat)

    kl_div = entropy(P_flat, Q_flat)
    is_collapsed = 1 if kl_div > EPSILON_THRESHOLD else 0
    return is_collapsed

# ==========================================
# 3. 閾値探索ループ
# ==========================================
print("シミュレーション開始...")
n_star_data = []
base_D = DIMENSIONS[0]
base_N = np.nan

for D in DIMENSIONS:
    valid_n = []
    for N_samp in SAMPLE_SIZES:
        results = Parallel(n_jobs=-1)(delayed(simulate_fast)(D, N_samp) for _ in range(N_TRIALS))
        rate = sum(results) / N_TRIALS
        if rate < 0.05:
            valid_n.append(N_samp)

    n_star = min(valid_n) if valid_n else np.nan
    if D == base_D:
        base_N = n_star

    theoretical_N = base_N * (D / base_D) if not np.isnan(base_N) else np.nan
    deviation = ((n_star - theoretical_N) / theoretical_N) * 100 if not np.isnan(theoretical_N) and not np.isnan(n_star) else np.nan

    n_star_data.append({
        "Dimension_D": D,
        "Simulated_N_star": n_star,
        "Theoretical_N_star": theoretical_N,
        "Deviation_Percentage": deviation
    })

df = pd.DataFrame(n_star_data)
df.to_csv(f"{output_dir}/threshold_analysis.csv", index=False)

# ==========================================
# 4. 可視化と保存
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(df["Dimension_D"], df["Deviation_Percentage"], marker='o', color='red', linewidth=2)
plt.axhline(y=20, color='gray', linestyle='--', label='20% Deviation (Warning Zone)')
plt.axhline(y=50, color='black', linestyle='--', label='50% Deviation (Critical Explosion)')
plt.xlabel("Categorical Dimension D")
plt.ylabel("Deviation from Linear Baseline (%)")
plt.title("Identification of Non-Linear Explosion Threshold")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/deviation_threshold.png", dpi=300)
plt.close()

zip_filename = "threshold_analysis.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            zipf.write(os.path.join(root, file), arcname=file)

files.download(zip_filename)

シミュレーション開始...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>